<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 5 · Buscar y cruzar sin #N/D
**Excel Corporativo · 12 Claves** — Lo que se pierde cuando un cruce falla · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/05_busqueda_y_cruce.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-05)

**Al terminar podrás:**
1. Ver cuántas filas de una exportación cruzan con el maestro y por qué fallan las demás.
2. Medir con un gráfico cuánto dinero queda sin clasificar si se cruza mal.
3. Comparar el resultado antes y después de limpiar la llave.
4. Decidir qué hacer con los códigos que no existen en el maestro.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · El problema
La exportación del ERP trae el **código del cliente**, pero no su **segmento**. Para el comité hay que mostrar la venta por segmento, así que hay que traer el segmento desde el maestro de clientes. En Excel eso se hace con **BUSCARX**; en Colab, con una unión de tablas. La lógica es la misma: buscar la llave en el maestro y traer el dato.

In [ ]:
# @title Cargar la exportación y mirar las llaves de cerca
erp = pd.read_csv(ruta("ventas_erp_sucio.csv"), dtype=str, keep_default_na=False)
def a_pesos(t):
    t = t.replace("$", "").replace(".", "").strip()
    return float(t) if t else np.nan
erp["valor"] = erp["Valor Neto"].map(a_pesos)
erp = erp.drop_duplicates(subset=["Linea"])
ids = set(clientes["id_cliente"])
muestra = erp[erp["Cod Cliente"].str.len() != 6].head(6)[["Factura", "Cod Cliente"]].copy()
muestra["Así se ve"] = muestra["Cod Cliente"]
muestra["Lo que realmente tiene"] = muestra["Cod Cliente"].map(lambda c: c.replace(" ", "·"))
muestra["Caracteres"] = muestra["Cod Cliente"].str.len()
display(muestra.drop(columns="Cod Cliente").style.hide(axis="index"))
display(Markdown("Un código válido tiene **6 caracteres** (`C-0129`). Los puntos `·` son espacios que en pantalla no se ven. En Excel, `=LARGO(A2)` los delata."))

## 2 · ¿Por qué falla cada fila?

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuántas filas cruzan y por qué fallan las demás? |
> | **Qué se grafica** | Barras: número de filas por motivo · Colores: cruza (azul), se arregla limpiando (teal), no existe (magenta) |
> | **Por qué esta forma** | Barras ordenadas: el motivo más frecuente salta a la vista; el color separa lo que se corrige de lo que se reporta. |
> | **Qué se descartó** | Torta: la parte que cruza es tan grande que las fallas se volverían rebanadas invisibles. |
> | **Cómo se lee** | Las barras teal se arreglan con una fórmula. La magenta no: hay que avisar al área de datos maestros. |
> | **En Excel 365** | Columna con BUSCARX y «No encontrado», luego CONTAR.SI por causa. |

In [ ]:
# @title Gráfico 1 · Motivos de fallo
def motivo(c):
    if c in ids: return "Cruza tal como viene"
    if c.strip() in ids: return "Espacios"
    if c.upper() in ids: return "Minúsculas"
    if c.strip().upper() in ids: return "Espacios y minúsculas"
    return "No existe en el maestro"
erp["motivo"] = erp["Cod Cliente"].map(motivo)
mot = erp["motivo"].value_counts().sort_values()
fallas = mot.drop("Cruza tal como viene")
colores = {"Cruza tal como viene": AZUL, "No existe en el maestro": MAGENTA}
fig = go.Figure(go.Bar(x=mot.values, y=mot.index, orientation="h", marker_color=[colores.get(k, TEAL) for k in mot.index],
                       text=mot.values, textposition="outside", cliponaxis=False))
fig.update_layout(title=f"{porc(mot['Cruza tal como viene'] / mot.sum())} cruza; la causa de fallo más común es «{fallas.idxmax()}»",
                  xaxis_title="Filas de la exportación", height=340)
dar_aire(fig, mot.max())
fig.show()
display(Markdown("En **Excel**, las filas con *Minúsculas* sí habrían cruzado: BUSCARX no distingue mayúsculas. Python sí."))

## 3 · Limpiar la llave cambia el resultado
Elige el nivel de limpieza y vuelve a ejecutar. Mira cómo cambian la tasa de cruce y el dinero sin clasificar.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánta venta queda sin segmento según cómo se limpie la llave? |
> | **Qué se grafica** | Barras: venta por segmento + una barra «Sin clasificar» · Filtro: nivel de limpieza |
> | **Por qué esta forma** | La barra magenta hace visible el dinero que el informe perdería; las demás muestran el reparto real. |
> | **Qué se descartó** | Mostrar solo la tasa de cruce: un 85 % suena bien hasta que se ve cuánto dinero es el 15 %. |
> | **Cómo se lee** | Mientras más corta la barra magenta, más confiable el informe por segmento. |
> | **En Excel 365** | Tabla dinámica con la columna del BUSCARX en Filas: «No encontrado» aparece como una categoría más. |

In [ ]:
# @title Gráfico 2 · Venta por segmento según la limpieza
limpieza = "Sin limpiar"  # @param ["Sin limpiar", "Quitar espacios", "Quitar espacios y pasar a mayúsculas"]
llave = erp["Cod Cliente"]
if limpieza != "Sin limpiar": llave = llave.str.strip()
if limpieza == "Quitar espacios y pasar a mayúsculas": llave = llave.str.upper()
seg = llave.map(clientes.set_index("id_cliente")["segmento"]).fillna("Sin clasificar")
tasa = (seg != "Sin clasificar").mean()
vv = erp[erp["Estado"] != "Anulada"].assign(segmento=seg).groupby("segmento")["valor"].sum().sort_values() / 1e6
fig = go.Figure(go.Bar(x=vv.values, y=vv.index, orientation="h", marker_color=[MAGENTA if k == "Sin clasificar" else AZUL for k in vv.index],
                       text=[f"$ {v:,.0f} M".replace(",", ".") for v in vv.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"{limpieza}: cruza el {porc(tasa)} de las filas y quedan {millones(vv.get('Sin clasificar', 0) * 1e6)} sin segmento",
                  xaxis_title=MM, height=360)
dar_aire(fig, vv.max())
fig.show()

## 4 · Lo que ninguna limpieza arregla
Los códigos que siguen sin cruzar **no existen en el maestro**: son clientes creados en el ERP que nadie registró. No se inventa su segmento ni se borran sus filas. Se entregan en una lista al área de datos maestros y, mientras tanto, se muestran como «Sin clasificar».

In [ ]:
# @title Ver la lista de códigos huérfanos
llave_ok = erp["Cod Cliente"].str.strip().str.upper()
tasa_limpia = llave_ok.isin(ids).mean()
huer = erp[~llave_ok.isin(ids)].groupby(llave_ok[~llave_ok.isin(ids)]).agg(filas=("Factura", "size"), venta=("valor", "sum"))
huer = huer.sort_values("venta", ascending=False).rename_axis("código")
display(Markdown(f"**{len(huer)} códigos** no existen en el maestro. Con la llave limpia cruza el **{porc(tasa_limpia)}** de las filas."))
display(huer.head(10).style.format({"venta": pesos}))

## 5 · Un maestro con duplicados multiplica filas
Si un código aparece dos veces en el maestro, BUSCARX devuelve la primera coincidencia **sin avisar**. En Python, la unión repite la fila de la venta por cada coincidencia, y los totales crecen. Por eso se comprueba que la llave del maestro sea única antes de cruzar.

In [ ]:
# @title Gráfico 3 · Qué pasa con un maestro duplicado
frecuente = llave_ok.value_counts().index[0]
dup = pd.concat([clientes, clientes[clientes.id_cliente == frecuente]])
filas_mal = len(erp.assign(llave=llave_ok).merge(dup[["id_cliente", "segmento"]], left_on="llave", right_on="id_cliente", how="left"))
fig = go.Figure(go.Bar(x=["Filas de la exportación", "Filas después de cruzar con el maestro duplicado"], y=[len(erp), filas_mal],
                       marker_color=[AZUL, MAGENTA], text=[f"{len(erp):,}".replace(",", "."), f"{filas_mal:,}".replace(",", ".")], textposition="outside"))
fig.update_layout(title=f"Duplicar un solo cliente ({frecuente}) en el maestro agrega {filas_mal - len(erp)} filas falsas a la venta", height=360, yaxis_title="Filas")
dar_aire(fig, filas_mal, "y")
fig.show()

## 6 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Cuál es la causa de fallo más común en esta exportación?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Espacios", "Minúsculas", "Espacios y minúsculas", "No existe en el maestro"]
revisar(respuesta, fallas.idxmax(), """Es la barra teal más larga del gráfico 1. Se corrige con ESPACIOS en Excel.""")

In [ ]:
# @title Reto 2 · Con la llave limpia, ¿qué porcentaje de filas cruza?
respuesta = "Elige una opción"  # @param ["Elige una opción", "84 %", "90 %", "98 %", "100 %"]
revisar(respuesta, f"{tasa_limpia:.0%}".replace("%", " %"), """Lo que falta para el 100 % son los códigos huérfanos.""")

In [ ]:
# @title Reto 3 · ¿Qué haces con los códigos que no existen en el maestro?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Asignarles el segmento más probable", "Borrar esas filas", "Reportarlos y mostrarlos como Sin clasificar", "Usar BUSCARV con coincidencia aproximada"]
revisar(respuesta, "Reportarlos y mostrarlos como Sin clasificar", """No se inventa ni se oculta: se reporta y se hace visible.""")

## Apóyate en IA
Cuando un cruce falle en Excel, pide un diagnóstico, no una fórmula nueva:

```text
Mi BUSCARX en Excel 365 devuelve "No encontrado" en el 15 % de las filas.
Estos son 10 códigos que no cruzaron: [pega aquí]. Estos son 5 códigos válidos del maestro: [pega aquí].
Dime las causas probables (espacios, carácter 160, ceros a la izquierda, códigos inexistentes)
y una fórmula de Excel en español para comprobar cada una.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Un cruce termina cuando sabes cuántas filas cruzaron y por qué fallaron las demás.
* Un porcentaje de cruce alto puede esconder mucho dinero: mídelo en pesos.
* Los huérfanos se reportan; un maestro con duplicados se corrige antes de cruzar.

**Siguiente:** [Clave 7 · Tablas dinámicas](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-07) · [Cuaderno 07](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/07_tablas_dinamicas.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>